In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch" 

from functools import partial
from molexpress import layers
from molexpress.datasets import featurizers
from molexpress.datasets import encoders
from molexpress.ops.chem_ops import get_molecule
import torch
import pandas as pd 
import pytorch_lightning as pl
import wandb
from torch.utils.data import random_split, DataLoader
from functools import partial
from tqdm import tqdm


In [2]:

atom_featurizers = [
    featurizers.AtomType(vocab={'C', 'N', 'O'}),
    featurizers.Hybridization(),
]

bond_featurizers = [
    featurizers.BondType(),
    featurizers.Conjugated()
]

peptide_graph_encoder = encoders.PeptideGraphEncoder(
    atom_featurizers=atom_featurizers, 
    bond_featurizers=bond_featurizers,
    self_loops=False, # self_loops True adds one feature dim to edge state
    supports_masking=True, # supports_masking True adds one feature dim to node and edge state
)

# Graph Neural Network using PyTorch Lightning
class GraphNeuralNetwork(torch.nn.Module):
    
    def __init__(self, dim):
        super().__init__()
        self.gcn1 = layers.GINConv(dim)
        self.gcn2 = layers.GINConv(dim)
        self.gcn3 = layers.GINConv(dim)
        self.gcn4 = layers.GINConv(dim)
        
    def forward(self, x):
        x = self.gcn1(x)
        x = self.gcn2(x)
        x = self.gcn3(x)
        x = self.gcn4(x)
        return x


class NodePrediction(torch.nn.Module):
    
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_dim, input_dim) 
        self.linear2 = torch.nn.Linear(input_dim, output_dim) 
        
    def forward(self, x):
        x = self.linear1(x['node_state'])
        x = torch.nn.functional.relu(x, inplace=False)
        x = self.linear2(x)
        return x


class EdgePrediction(torch.nn.Module):
    
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_dim, input_dim) 
        self.linear2 = torch.nn.Linear(input_dim, output_dim)
        self.gather_incident = layers.GatherIncident()
        
    def forward(self, x):
        x = self.gather_incident(x) # We do not use edge states but incident node states.
        x = self.linear1(x)
        x = torch.nn.functional.relu(x, inplace=False)
        x = self.linear2(x)
        return x


class GraphDataset(torch.utils.data.Dataset):
    
    def __init__(self, x):
        self.x = x

    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, index):
        graph = peptide_graph_encoder(self.x[index])
        return graph


class GraphModelModule(pl.LightningModule):
    
    def __init__(self, graph_model, node_pred_model, edge_pred_model, lr=0.001):
        super().__init__()
        self.graph_model = graph_model
        self.node_pred_model = node_pred_model
        self.edge_pred_model = edge_pred_model
        self.loss_fn = torch.nn.BCELoss(reduction='none')
        self.lr = lr

    def forward(self, x):
        graph = self.graph_model(x)
        node_pred = self.node_pred_model(graph)
        edge_pred = self.edge_pred_model(graph)
        return node_pred, edge_pred

    def training_step(self, batch, batch_idx):
        graph = self.graph_model(batch)
        node_pred = self.node_pred_model(graph)
        edge_pred = self.edge_pred_model(graph)
        
        node_loss = self.weighted_loss(node_pred, batch['node_label'], batch['node_loss_weight'])
        edge_loss = self.weighted_loss(edge_pred, batch['edge_label'], batch['edge_loss_weight'])
        loss = node_loss + edge_loss

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True,batch_size = 1024)
        return loss

    def validation_step(self, batch, batch_idx):
        graph = self.graph_model(batch)
        node_pred = self.node_pred_model(graph)
        edge_pred = self.edge_pred_model(graph)
        
        node_loss = self.weighted_loss(node_pred, batch['node_label'], batch['node_loss_weight'])
        edge_loss = self.weighted_loss(edge_pred, batch['edge_label'], batch['edge_loss_weight'])
        loss = node_loss + edge_loss

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True,batch_size = 1024)
        return loss

 
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            list(self.graph_model.parameters()) + 
            list(self.node_pred_model.parameters()) + 
            list(self.edge_pred_model.parameters()), 
            lr=self.lr,weight_decay=0.001
        )
        return optimizer

    def weighted_loss(self, pred, true, weight):
        log = torch.sigmoid(pred)    # Sigmoid() only with BCELoss
        true = torch.from_numpy(true)
        true  = true.to("cuda")
        weight = torch.from_numpy(weight)
        weight = weight.to("cuda")
        assert true.shape ==log.shape, f"Expected the two inputs to have the same shape"
   
        loss = self.loss_fn(log, true)
        # print(true.get_device(),log.get_device(),loss.get_device(),)
        w_loss = loss * weight[:, None]  # weight[:, None] only with BCELoss
        return torch.mean(w_loss)

    


In [3]:
# checkpoint = torch.load("model_checkpoint_shuffled_adamW.ckpt")
# state_dict = checkpoint["state_dict"]
state_dict = torch.load("model_checkpoint_shuffled_adamW.ckpt")
remove_prefix = 'module.'
state_dict = {k[len(remove_prefix):] if k.startswith(remove_prefix) else k: v for k, v in state_dict.items()}


/tmp/ipykernel_3479419/3684643767.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("model_checkpoint_shuffled_adamW.ckpt")


In [4]:
state_dict

{'epoch': 250,
 'global_step': 976750,
 'pytorch-lightning_version': '2.4.0',
 'state_dict': OrderedDict([('graph_model.gcn1._torch_params.gin_conv/epsilon',
               tensor(0.)),
              ('graph_model.gcn1._torch_params.gin_conv/node_bias_1',
               tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
                       0., 0., 0., 0., 0., 0., 0., 0.])),
              ('graph_model.gcn1._torch_params.gin_conv/node_bias_2',
               tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
                       0., 0., 0., 0., 0., 0., 0., 0.])),
              ('graph_model.gcn1._torch_params.gin_conv/node_kernel_2',
               tensor([[-0.1198, -0.2899,  0.2831,  ...,  0.2490,  0.1575, -0.0075],
                       [-0.1528,  0.2869, -0.1900,  ..., -0.1552,  0.1281,  0.2691],
                       [-0.2827, -0.3045, -0.2597,  ...,  0.0285,  0.2153, -0.

In [5]:
state_dict

{'epoch': 250,
 'global_step': 976750,
 'pytorch-lightning_version': '2.4.0',
 'state_dict': OrderedDict([('graph_model.gcn1._torch_params.gin_conv/epsilon',
               tensor(0.)),
              ('graph_model.gcn1._torch_params.gin_conv/node_bias_1',
               tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
                       0., 0., 0., 0., 0., 0., 0., 0.])),
              ('graph_model.gcn1._torch_params.gin_conv/node_bias_2',
               tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
                       0., 0., 0., 0., 0., 0., 0., 0.])),
              ('graph_model.gcn1._torch_params.gin_conv/node_kernel_2',
               tensor([[-0.1198, -0.2899,  0.2831,  ...,  0.2490,  0.1575, -0.0075],
                       [-0.1528,  0.2869, -0.1900,  ..., -0.1552,  0.1281,  0.2691],
                       [-0.2827, -0.3045, -0.2597,  ...,  0.0285,  0.2153, -0.

In [6]:
graph_model = GraphNeuralNetwork(32)
node_pred_model = NodePrediction(32, 11)
edge_pred_model = EdgePrediction(32 * 2, 6)

model = GraphModelModule.load_from_checkpoint(
    "model_checkpoint_shuffled_adamW.ckpt",
    graph_model=graph_model,
    node_pred_model=node_pred_model,
    edge_pred_model=edge_pred_model,
    strict=False
      )





/home/harikrishnan/miniconda3/envs/molexp/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['graph_model.gcn1._torch_params.gin_conv/epsilon', 'graph_model.gcn1._torch_params.gin_conv/node_bias_1', 'graph_model.gcn1._torch_params.gin_conv/node_bias_2', 'graph_model.gcn1._torch_params.gin_conv/node_kernel_2', 'graph_model.gcn1._torch_params.gin_conv/special_edge_kernel', 'graph_model.gcn1._torch_params.gin_conv/special_node_kernel', 'graph_model.gcn1.normalize._torch_params.gin_conv/batch_normalization/beta', 'graph_model.gcn1.normalize._torch_params.gin_conv/batch_normalization/gamma', 'graph_model.gcn1.normalize._torch_params.gin_conv/batch_normalization/moving_mean', 'graph_model.gcn1.normalize._torch_params.gin_conv/batch_normalization/moving_variance', 'graph_model.gcn2._torch_params.gin_conv_1/epsilon', 'graph_model.gcn2._torch_params.gin_conv_1/node_bias_1', 'graph_model.gcn2._torch_params.gi

In [7]:

data = pd.read_csv("/home/harikrishnan/molexpress-main/molexpress/pretraining/smiles_finetune.csv",names=["smiles"])
data = data.sample(frac=1)
dataset = data["smiles"].apply(lambda x: [x]).to_list()

dataset

[['C12=C3C(=CC=C1C=CC4=C2C=CC=C4)C=CC=C3'],
 ['C[C@H](NC(C)=O)C(O)=O'],
 ['Nc1nc2n(cnc2c(=O)[nH]1)[C@@H]1O[C@H](COP([O-])(=O)OP([O-])(=O)OP([O-])(=O)OP([O-])(=O)OC[C@H]2O[C@H]([C@H](O)[C@@H]2O)n2cnc3c2nc(N)[nH]c3=O)[C@@H](O)[C@H]1O'],
 ['[O-]P([O-])(=O)OC[C@H](N-*)C(-*)=O'],
 ['COc1ccc(CC=C)cc1OC'],
 ['COC(=O)[C@@H](N)CC(C)C'],
 ['[H]C(*)=O'],
 ['N[C@@H](Cc1ccc(O)c(O)c1)C(O)=O'],
 ['C[S@@](=O)CC[C@H]([NH3+])C([O-])=O'],
 ['NCCc1c[nH]cn1'],
 ['NC(Cc1ccc(O)c(c1)-c1cc(CC(N)C(O)=O)ccc1O)C(O)=O'],
 ['CC(=O)NCC(O)=O'],
 ['Cc1cc2c(O)c(O)ccc2[nH]c1=O'],
 ['N[C@@H](CC(O)=O)C(N)=O'],
 ['N[C@@H](Cc1ccc(O)c(Br)c1)C(O)=O'],
 ['CCC(C)C(NC(C)=O)C(O)=O'],
 ['C1(=CNC2=C1C=C(C=C2)Cl)C[C@@H](C(=O)O)N'],
 ['[Cu++].[O-][N+]([O-])=O.[O-][N+]([O-])=O'],
 ['C(C(C(O)=O)N)C1=CC=CC=C1F'],
 ['C[N+](C)(C)CC(O)O'],
 ['NCC(O)=O'],
 ['NC(Cc1ccc(F)cc1)C(O)=O'],
 ['CC1CCCCOC1=O'],
 ['NC1CC(=O)N(CC(O)=O)C1=O'],
 ['C[C@@H](SC[C@H](N-*)C(-*)=O)C1=C(C)C(=O)N[C@H]1CC1=N\\C(=C/c2[nH]c(CC3NC(=O)C(C=C)=C3C)c(C)c2CCC([O-])=O)C(

In [8]:

dataset = GraphDataset(dataset)



In [9]:
test_elem = dataset.__getitem__(1)

In [10]:

data = pd.read_csv("/home/harikrishnan/molexpress-main/molexpress/pretraining/canon_filtered_pubchem.txt",names=["smiles"])
# data = data.sample(frac=1)
dataset = data["smiles"].apply(lambda x: [x]).to_list()


# Validation split and DataLoaders
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size],generator= torch.Generator().manual_seed(42))


batch_size = 1024
partial_collate_fn = partial(
    peptide_graph_encoder.masked_collate_fn, node_masking_rate=0.3, edge_masking_rate=0.3
)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size],generator= torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=partial_collate_fn)


In [11]:

for ind, data in enumerate(train_loader):
    print(data)
    print(ind)


TypeError: list indices must be integers or slices, not str